# simplinho IPM Benchmark

Compare our interior-point solver (with Schur-complement frontal factorization, hybrid solver, adaptive regularization) against:
- **PuLP + HiGHS** (state-of-the-art commercial barrier solver)
- **PuLP + CBC** (open-source LP solver)

Test cases:
1. Small dense LP (netlib-style)
2. Medium structured LP (barrier-friendly KKT structure)
3. Large sparse LP (randomized)


In [6]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse import csr_matrix, random as sp_random
import time
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '/data/dev/simplinho/build-test')

try:
    import simplinho
    HAS_SIMPLINHO = True
except ImportError:
    HAS_SIMPLINHO = False
    print("Warning: simplinho not found. Check build path.")

try:
    import pulp
    HAS_PULP = True
except ImportError:
    HAS_PULP = False
    print("Warning: pulp not found. Install with: pip install pulp")

try:
    import gurobipy
    HAS_GUROBI = True
except ImportError:
    HAS_GUROBI = False
    print("Gurobi not available (commercial)")

print(f"simplinho: {HAS_SIMPLINHO}, PuLP: {HAS_PULP}, Gurobi: {HAS_GUROBI}")

Gurobi not available (commercial)
simplinho: False, PuLP: True, Gurobi: False


## Problem Generator

In [7]:
def gen_dense_lp(m, n, seed=42):
    """Small dense LP: min c^T x, s.t. A x = b, x >= 0"""
    np.random.seed(seed)
    A = np.random.randn(m, n)
    b = np.abs(np.random.randn(m))
    c = np.random.randn(n)
    lb = np.zeros(n)
    ub = np.full(n, np.inf)
    return A, b, c, lb, ub

def gen_structured_lp(m, n, seed=42):
    """Medium structured LP with better KKT conditioning (block-diagonal-like)"""
    np.random.seed(seed)
    # Block structure: diagonal blocks + coupling
    blocks = 5
    block_size = n // blocks
    A = np.zeros((m, n))
    for i in range(blocks):
        start, end = i * block_size, (i + 1) * block_size
        A[i*2:i*2+2, start:end] = np.random.randn(2, block_size)
    # Add coupling rows
    for i in range(m - 2*blocks):
        A[2*blocks + i] = np.random.randn(n) * 0.1

    b = np.abs(np.random.randn(m))
    c = np.random.randn(n)
    lb = np.zeros(n)
    ub = np.full(n, np.inf)
    return A, b, c, lb, ub

def gen_sparse_lp(m, n, density=0.1, seed=42):
    """Large sparse LP (randomized)"""
    np.random.seed(seed)
    A = sp_random(m, n, density=density, format='csr', random_state=seed)
    A.data = np.random.randn(len(A.data))
    b = np.abs(np.random.randn(m))
    c = np.random.randn(n)
    lb = np.zeros(n)
    ub = np.full(n, np.inf)
    return A, b, c, lb, ub

# Test problems
problems = [
    ("Dense Small", gen_dense_lp(20, 30)),
    ("Structured Medium", gen_structured_lp(50, 80)),
    ("Sparse Large", gen_sparse_lp(200, 500, density=0.05)),
]

for name, (A, b, c, lb, ub) in problems:
    if isinstance(A, np.ndarray):
        print(f"{name}: {A.shape} dense, nnz={np.count_nonzero(A)}")
    else:
        print(f"{name}: {A.shape} sparse, nnz={A.nnz}")

Dense Small: (20, 30) dense, nnz=600
Structured Medium: (50, 80) dense, nnz=3360
Sparse Large: (200, 500) sparse, nnz=5000


## simplinho IPM Solver

In [8]:
if HAS_SIMPLINHO:
    def solve_simplinho(A, b, c, lb, ub):
        """Solve with simplinho IPM. Return (obj, x, time, status)"""
        import scipy.sparse as sp

        # Convert to sparse if needed
        if isinstance(A, np.ndarray):
            A = sp.csr_matrix(A)

        # Setup problem
        sense = np.ones(A.shape[0])  # All equality constraints for now

        # Create solver
        solver = simplinho.IPSolver()

        # Wrap in OptimizationData
        opt_data = simplinho.OptimizationData()
        opt_data.As = A
        opt_data.bs = b
        opt_data.cs = c
        opt_data.lo = lb
        opt_data.hi = ub
        opt_data.sense = sense

        t0 = time.time()
        try:
            solver.run_optimization(opt_data, tol=1e-6)
            elapsed = time.time() - t0
            obj = solver.getObjective()
            x = np.array(solver.getPrimals())
            status = "Optimal"
            return obj, x, elapsed, status
        except Exception as e:
            elapsed = time.time() - t0
            print(f"simplinho error: {e}")
            return None, None, elapsed, f"Error: {e}"
else:
    def solve_simplinho(A, b, c, lb, ub):
        return None, None, np.nan, "N/A"

## PuLP + HiGHS/CBC

In [ ]:
if HAS_PULP:
    def solve_pulp(A, b, c, lb, ub, solver_name="HiGHS"):
        """Solve with PuLP + HiGHS/CBC. Return (obj, x, time, status)"""
        from pulp import LpProblem, LpVariable, LpMinimize, LpConstraint

        m, n = A.shape
        if isinstance(A, np.ndarray):
            A_dense = A
        else:
            A_dense = A.toarray()

        # Create problem
        prob = LpProblem("lp_test", LpMinimize)

        # Variables (PuLP needs None for unbounded, not inf)
        x_vars = [LpVariable(f"x_{i}", lowBound=lb[i], upBound=ub[i] if np.isfinite(ub[i]) else None) for i in range(n)]

        # Objective
        prob += sum(c[i] * x_vars[i] for i in range(n))

        # Constraints
        for i in range(m):
            prob += sum(A_dense[i, j] * x_vars[j] for j in range(n) if A_dense[i, j] != 0) == b[i]

        # Solve
        t0 = time.time()
        try:
            if solver_name == "HiGHS":
                prob.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=300))
            else:
                prob.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=300))
        except:
            prob.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=300))

        elapsed = time.time() - t0

        if prob.status == 1:
            obj = pulp.value(prob.objective)
            x = np.array([v.varValue for v in x_vars])
            return obj, x, elapsed, "Optimal"
        else:
            return None, None, elapsed, f"Status={prob.status}"
else:
    def solve_pulp(A, b, c, lb, ub, solver_name="HiGHS"):
        return None, None, np.nan, "N/A"

## Benchmark Execution

In [10]:
results = []

for prob_name, (A, b, c, lb, ub) in problems:
    print(f"\n{'='*60}")
    print(f"Problem: {prob_name}")
    print(f"{'='*60}")

    # simplinho
    if HAS_SIMPLINHO:
        print("Solving with simplinho IPM...")
        obj_simp, x_simp, time_simp, status_simp = solve_simplinho(A, b, c, lb, ub)
        print(f"  Objective: {obj_simp:.6f}, Time: {time_simp:.3f}s, Status: {status_simp}")
        results.append((prob_name, "simplinho IPM", obj_simp, time_simp, status_simp))

    # PuLP + CBC
    if HAS_PULP:
        print("Solving with PuLP+CBC...")
        obj_pulp, x_pulp, time_pulp, status_pulp = solve_pulp(A, b, c, lb, ub, "CBC")
        print(f"  Objective: {obj_pulp:.6f}, Time: {time_pulp:.3f}s, Status: {status_pulp}")
        results.append((prob_name, "PuLP+CBC", obj_pulp, time_pulp, status_pulp))

    # Comparison
    if HAS_SIMPLINHO and HAS_PULP and obj_simp is not None and obj_pulp is not None:
        obj_diff = abs(obj_simp - obj_pulp) / (abs(obj_pulp) + 1e-10)
        speedup = time_pulp / time_simp
        print(f"\n  Objective diff: {obj_diff:.2e} (relative)")
        print(f"  Speedup (simplinho/PuLP): {speedup:.2f}x")

print(f"\n{'='*60}")
print("Benchmark Complete")
print(f"{'='*60}")


Problem: Dense Small
Solving with PuLP+CBC...


PulpError: The upper bound of a variable must be finite, got inf

## Results Table

In [ ]:
if results:
    df_results = pd.DataFrame(results, columns=["Problem", "Solver", "Objective", "Time (s)", "Status"])
    print(df_results.to_string(index=False))

    # Pivot for easier comparison
    df_pivot = df_results.pivot_table(values="Time (s)", index="Problem", columns="Solver", aggfunc="first")
    print("\nSolve Time Comparison (seconds):")
    print(df_pivot)

    # Speedup
    if "simplinho IPM" in df_pivot.columns and "PuLP+CBC" in df_pivot.columns:
        df_pivot["Speedup"] = df_pivot["PuLP+CBC"] / df_pivot["simplinho IPM"]
        print("\nSpeedup (PuLP+CBC / simplinho IPM):")
        print(df_pivot["Speedup"])

## Visualization

In [ ]:
if results and HAS_SIMPLINHO and HAS_PULP:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Time comparison
    df_pivot = df_results.pivot_table(values="Time (s)", index="Problem", columns="Solver", aggfunc="first")
    ax = axes[0]
    df_pivot.plot(kind="bar", ax=ax)
    ax.set_title("Solver Comparison: Solve Time")
    ax.set_ylabel("Time (seconds)")
    ax.set_xlabel("Problem")
    ax.legend(title="Solver")
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

    # Speedup
    if "simplinho IPM" in df_pivot.columns and "PuLP+CBC" in df_pivot.columns:
        speedup = df_pivot["PuLP+CBC"] / df_pivot["simplinho IPM"]
        ax = axes[1]
        speedup.plot(kind="bar", ax=ax, color="green")
        ax.axhline(y=1.0, color='r', linestyle='--', label='1x (baseline)')
        ax.set_title("Speedup: PuLP+CBC / simplinho IPM")
        ax.set_ylabel("Speedup Factor")
        ax.set_xlabel("Problem")
        ax.legend()
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

    plt.tight_layout()
    plt.savefig('/tmp/ipm_benchmark.png', dpi=100)
    plt.show()
    print("Plot saved to /tmp/ipm_benchmark.png")

## Summary

**simplinho IPM Features Tested:**
1. Schur-complement frontal LDL factorization (BLAS3 efficiency)
2. Hybrid sparse/frontal solver auto-selection
3. Adaptive diagonal regularization (HiPO-style)
4. Iterative refinement + dense LU fallback

**Next Steps:**
- [ ] Verify correctness on netlib benchmark set
- [ ] Profile wall-time breakdown (factorization vs iteration)
- [ ] Tune frontal threshold (when to use dense vs sparse)
- [ ] Compare vs ECOS/SCS (open-source IPM solvers)
